# 5.1 마진 최대화와 서포트 벡터 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter05_1_svm_margin.ipynb)

책 본문: [5.1 마진 최대화와 서포트 벡터](https://smhanlab.com/book-ml/kor/ml1/chapter05/1.html)

이 노트북은 책 5.1절의 **모든 손 계산값을 코드로 검증**합니다:

1. 대칭 데이터에서 손으로 푼 최적해 \(w=(1/6,1/6),\ b=0\)과 sklearn SVM이 일치하는지
2. 스케일 불변성 — 같은 경계를 \(k\)배로 쓰면 기능적 마진이 \(k\)배, 기하적 마진은 그대로인지
3. 데이터 전체를 병진하면 \(w\)는 그대로, \(b\)만 변하는지
4. 결정 경계·마진 경계·서포트 벡터를 그림으로 보고, 경계에서 SV까지의
   실제 거리가 정확히 \(1/\|w\|\)인지 직접 재보기
5. 확인 문제 — 서포트 벡터 아닌 점을 경계 안쪽으로 당기면 \(w,b\)가 바뀌는지
6. 서포트 벡터 아닌 점을 멀리 밀어도 \(w,b\)가 안 변하는 SVM vs 로지스틱회귀 대조

## 1. 대칭 데이터: 본문의 손 계산 검증

본문 설정: 양성 \((3,3),(4,3),(3,4)\), 음성 \((-3,-3),(-4,-3),(-3,-4)\).
대칭성으로 \(b=0,\ w\propto(1,1)\)이고, 가장 가까운 점 \((3,3)\)이 마진
경계 위에 있다 → \(6k=1 \Rightarrow k=1/6\). 즉 \(w=(1/6,1/6),\ b=0\),
서포트 벡터는 \((3,3),(-3,-3)\) 두 점, 마진 \(2/\|w\|=12/\sqrt2\approx8.485\).

In [1]:
import numpy as np
from sklearn.svm import SVC

# 본문 "손으로 한 번"의 데이터
X_sym = np.array([[ 3,  3], [ 4,  3], [ 3,  4],
                  [-3, -3], [-4, -3], [-3, -4]], float)
y     = np.array([ 1,  1,  1, -1, -1, -1])

clf = SVC(kernel="linear", C=1.0).fit(X_sym, y)
w, b = clf.coef_[0], clf.intercept_[0]
print(f"w = {w}   b = {b}")
print(f"서포트 벡터: {X_sym[clf.support_].tolist()}")

# 본문 손 계산값과 비교
assert np.allclose(w, [1/6, 1/6]), w
assert abs(b) < 1e-9, b
assert set(map(tuple, X_sym[clf.support_].tolist())) == {(3.0, 3.0), (-3.0, -3.0)}

margin = 2 / np.linalg.norm(w)
print(f"마진 2/||w|| = {margin:.4f}   (12/sqrt(2) = {12/np.sqrt(2):.4f})")
assert abs(margin - 12/np.sqrt(2)) < 1e-9

# 본문 표: 각 점의 기능적 마진
print("\n점          기능적 마진 y*(w^Tx+b)   서포트 벡터?")
for xi, yi in zip(X_sym, y):
    fm = yi * (w @ xi + b)
    print(f"({xi[0]:+.0f},{xi[1]:+.0f})  {fm:.3f}  {'O (등호)' if abs(fm - 1) < 1e-9 else 'x (여유)'}")

w = [0.16666667 0.16666667]   b = -0.0
서포트 벡터: [[-3.0, -3.0], [3.0, 3.0]]
마진 2/||w|| = 8.4853   (12/sqrt(2) = 8.4853)

점          기능적 마진 y*(w^Tx+b)   서포트 벡터?
(+3,+3)  1.000  O (등호)
(+4,+3)  1.167  x (여유)
(+3,+4)  1.167  x (여유)
(-3,-3)  1.000  O (등호)
(-4,-3)  1.167  x (여유)
(-3,-4)  1.167  x (여유)


## 2. 스케일 불변성: 같은 경계, 다른 \(w\) — 기능적 vs 기하적 마진

\((w,b)\)와 \((kw,kb)\)는 **정확히 같은 결정 경계**다. 같은 경계를
\(3\)배 스케일로 다시 쓰면, 기능적 마진은 3배가 되지만 기하적 마진은
그대로다 — 실제 거리를 재는 것은 스케일에 무관한 기하적 마진뿐이다.
본문의 예: 최적해에서 서포트 벡터 \((3,3)\)는 기능적 마진 1, 기하적 마진
\(6/\sqrt2\approx4.243\).

In [2]:
def functional_margin(w, b, x, y):
    return y * (w @ x + b)

def geometric_margin(w, b, x, y):
    return functional_margin(w, b, x, y) / np.linalg.norm(w)

x_sv = np.array([3.0, 3.0]); y_sv = 1.0
w1 = np.array([1/6, 1/6]); b1 = 0.0     # 본문의 최적해
w2 = 3 * w1;             b2 = 0.0       # 같은 경계, 3배 스케일

print(f"경계 f(x)=w^Tx+b (w1): {w1} b={b1}")
print(f"경계 f(x)=w^Tx+b (w2): {w2} b={b2}")

for name, (wi, bi) in [("최적해 (1/6,1/6)", (w1, b1)), ("3배 스케일 (1/2,1/2)", (w2, b2))]:
    fm = functional_margin(wi, bi, x_sv, y_sv)
    gm = geometric_margin(wi, bi, x_sv, y_sv)
    print(f"{name:22s} 기능적 마진 = {fm:.3f}   기하적 마진 = {gm:.4f}")

# 같은 경계인지(점의 부호 불변) 확인
for xi, yi in zip(X_sym, y):
    assert np.sign(yi * (w1 @ xi + b1)) == np.sign(yi * (w2 @ xi + b2))
assert functional_margin(w2, b2, x_sv, y_sv) == 3 * functional_margin(w1, b1, x_sv, y_sv)
assert abs(geometric_margin(w2, b2, x_sv, y_sv) - geometric_margin(w1, b1, x_sv, y_sv)) < 1e-12
print("같은 경계: 부호 불변, 기능적 마진 3배, 기하적 마진 동일 -> 스케일 불변성 검증")

경계 f(x)=w^Tx+b (w1): [0.16666667 0.16666667] b=0.0
경계 f(x)=w^Tx+b (w2): [0.5 0.5] b=0.0
최적해 (1/6,1/6)          기능적 마진 = 1.000   기하적 마진 = 4.2426
3배 스케일 (1/2,1/2)       기능적 마진 = 3.000   기하적 마진 = 4.2426
같은 경계: 부호 불변, 기능적 마진 3배, 기하적 마진 동일 -> 스케일 불변성 검증


## 3. 병진 실험: 데이터 전체를 +2 이동하면 \(w\)는 그대로, \(b\)만 변한다

본문 예: 6개 데이터를 전부 x1 방향으로 +2 이동 → 양성 \((5,3),(6,3),(5,4)\),
음성 \((-1,-3),(-2,-3),(-1,-4)\). 최적해는 \(w=(1/6,1/6)\)(변함없음),
\(b=-1/3\), 서포트 벡터는 \((5,3),(-1,-3)\).

In [3]:
X_shift = X_sym.copy()
X_shift[:, 0] += 2.0
print(f"병진 후 데이터: {X_shift.tolist()}")

clf_s = SVC(kernel="linear", C=1.0).fit(X_shift, y)
w_s, b_s = clf_s.coef_[0], clf_s.intercept_[0]
print(f"w = {w_s}   b = {b_s}")
print(f"서포트 벡터: {X_shift[clf_s.support_].tolist()}")

# 본문 손 계산값과 비교
assert np.allclose(w_s, [1/6, 1/6]), "w가 변했으면 안 된다"
assert abs(b_s - (-1/3)) < 1e-9, b_s
assert set(map(tuple, X_shift[clf_s.support_].tolist())) == {(5.0, 3.0), (-1.0, -3.0)}

# 병진이 b를 어떻게 바꿨는지 직접 계산: x_new = x_old + t, 즉 x_old = x_new - t
# 경계 w^T x_old + b = 0  ->  w^T x_new + (b - w^T t) = 0  ->  b' = b - w^T t
t = np.array([2.0, 0.0])
print(f"b' = b - w^T t = 0 - {w @ t} = {-w @ t:.4f}  (sklearn의 b' = {b_s:.4f})")
assert abs((b - w @ t) - b_s) < 1e-9
print("w는 그대로, b만 -w^T t만큼 변한다 -> 병진은 방향과 무관한 '위치' 문제")

병진 후 데이터: [[5.0, 3.0], [6.0, 3.0], [5.0, 4.0], [-1.0, -3.0], [-2.0, -3.0], [-1.0, -4.0]]
w = [0.16666667 0.16666667]   b = -0.3333333333333334
서포트 벡터: [[-1.0, -3.0], [5.0, 3.0]]
b' = b - w^T t = 0 - 0.3333333333333333 = -0.3333  (sklearn의 b' = -0.3333)
w는 그대로, b만 -w^T t만큼 변한다 -> 병진은 방향과 무관한 '위치' 문제


## 4. 그림: 결정 경계, 마진 경계, 서포트 벡터

본문 "실습" 코드의 완성형. 대각선으로 떨어진 두 클러스터에 선형 SVM을
학습시키고, **결정 경계** \(f=0\), **마진 경계** \(f=\pm1\),
**서포트 벡터**를 한 그림에 겹쳐본다.

기울기가 \(\|w\|\)라서 \(f\)가 1만큼 변하는 데는 \(w\) 방향으로
\(1/\|w\|\)의 이동이 필요하다는 본문의 논리(3절)를, 이번엔
**직각 투영으로** 정확히 검증한다: 경계 위의 점 \(x_0\)에서 SV \(x_+\)까지
의 최단 거리는 \(\|x_+ - x_0\|\)가 아니라, 그 **수직 성분**이다 —
수직 성분의 크기가 정확히 \(1/\|w\|\)여야 한다. (경계 위의 점
\(x_0\)는 \(x_+ - (w^Tx_++b)/\|w\|^2\, w\)로 구한다:
\(x_+\)에서 \(w\) 방향으로 \((w^Tx_++b)/\|w\|^2\)만큼 되돌아가면
\(f=0\)이 된다.)

In [4]:
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

# 대각선 방향으로 떨어진 두 클러스터 (본문 코드 그대로)
rng = np.random.default_rng(7)
X = np.vstack([rng.normal([5, 5], 0.8, (40, 2)),
               rng.normal([-5, -5], 0.8, (40, 2))])
y_c = np.r_[np.ones(40), -np.ones(40)]

clf_c = SVC(kernel="linear", C=1.0).fit(X, y_c)
w_c, b_c = clf_c.coef_[0], clf_c.intercept_[0]
svs = X[clf_c.support_]

# f(x) = w^T x + b = c  ->  x2 = (-w1*x1 - b + c)/w2
x1 = np.linspace(-8, 8, 100)
f_line = lambda c: (-w_c[0] * x1 - b_c + c) / w_c[1]

fig, ax = plt.subplots(figsize=(6.5, 6.5))
for lbl, m in [(1, "red"), (-1, "blue")]:
    ax.scatter(X[y_c == lbl, 0], X[y_c == lbl, 1], color=m, alpha=0.6, label=f"y={lbl}")
ax.plot(x1, f_line(0),  "k",   lw=2,  label="결정 경계 (f=0)")
ax.plot(x1, f_line(1),  "g--", lw=1,  label="마진 경계 (f=+1)")
ax.plot(x1, f_line(-1), "g--", lw=1,  label="마진 경계 (f=-1)")
ax.scatter(svs[:, 0], svs[:, 1], s=160, facecolors="none",
           edgecolors="orange", lw=2.5, label="서포트 벡터")
ax.set_aspect("equal"); ax.legend(loc="upper left", fontsize=9)
ax.set_title("결정 경계, 마진 경계, 서포트 벡터")
fig.tight_layout()
fig.savefig("/home/smhan/book-ml/kor/src/images/ch05_1_svm_margin.svg")
plt.show()

print(f"서포트 벡터 개수: {len(svs)} / {len(X)}")
for sv in svs:
    print(f"  ({sv[0]:+.2f}, {sv[1]:+.2f})  (f = {w_c @ sv + b_c:+.3f})")
margin_c = 2 / np.linalg.norm(w_c)
print(f"마진 2/||w|| = {margin_c:.4f}")

서포트 벡터 개수: 2 / 80
  (-4.00, -3.85)  (f = -1.000)
  (+3.48, +3.97)  (f = +1.000)
마진 2/||w|| = 10.8176


## 5. 경계~서포트 벡터 거리를 직접 재본다: 정확히 \(1/\|w\|\)

서포트 벡터마다 (1) 마진 경계 위에 있는지(\(|f|\)가 1인지),
(2) 경계까지의 **수직 거리**가 \(1/\|w\|\)인지,
(3) **수평 거리**(경계와 나란한 성분)가 0인지 확인한다.
이 3가지를 전부 만족해야 "\(2/\|w\|\)가 실제 거리"라는 본문의
공식이 성립한다.

In [5]:
W2_c = w_c @ w_c
ok = True
for sv in svs:
    f_val = w_c @ sv + b_c                    # SV라면 1 또는 -1
    foot  = sv - f_val / W2_c * w_c           # sv의 경계(f=0) 직각 투영
    dist_v = np.linalg.norm(f_val / W2_c * w_c)  # 수직 성분 = f_val/||w||
    perp   = sv - foot                        # = 수직 성분 (수평 성분 없음)
    is_ok  = abs(abs(f_val) - 1) < 1e-6 and abs(dist_v - 1/np.linalg.norm(w_c)) < 1e-6  # 해의 수렴 허용치(약 1e-8)
    ok &= is_ok
    print(f"SV ({sv[0]:+.2f}, {sv[1]:+.2f}): |f|={abs(f_val):.6f}  "
          f"수직 거리={dist_v:.6f}  (1/||w||={1/np.linalg.norm(w_c):.6f})  {'O' if is_ok else 'X'}")
assert ok
print("모든 SV가 마진 경계 위에 있고, 경계까지의 실제 거리가 정확히 1/||w||")

SV (-4.00, -3.85): |f|=1.000000  수직 거리=5.408795  (1/||w||=5.408795)  O
SV (+3.48, +3.97): |f|=1.000000  수직 거리=5.408795  (1/||w||=5.408795)  O
모든 SV가 마진 경계 위에 있고, 경계까지의 실제 거리가 정확히 1/||w||


## 6. 확인 문제: 서포트 벡터 아닌 점을 당기면 \(w,b\)가 바뀐다

본문 "확인 문제": \((4,3)\)을 \((0.5,0.5)\)로 당기면(라벨 +1 그대로),
이 점이 새 마진 제약 \(y(w^Tx+b)\ge1\)을 위반하는 후보가 되어 경계가
바뀌어야 한다 — 본문 예상값 \(w\approx(0.286,0.286),\ b\approx0.714\),
새 서포트 벡터 \((0.5,0.5),(-3,-3)\).

In [6]:
X_q = X_sym.copy()
X_q[1] = [0.5, 0.5]    # (4,3) -> (0.5,0.5)
clf_q = SVC(kernel="linear", C=1.0).fit(X_q, y)
w_q, b_q = clf_q.coef_[0], clf_q.intercept_[0]
print(f"w = {w_q}   b = {b_q}")
print(f"서포트 벡터: {X_q[clf_q.support_].tolist()}")

# 본문 예상값과 비교
assert np.allclose(w_q, [0.28571429, 0.28571429], atol=1e-6), w_q
assert abs(b_q - 0.71428571) < 1e-6, b_q
assert set(map(tuple, X_q[clf_q.support_].tolist())) == {(0.5, 0.5), (-3.0, -3.0)}
print("본문 예상값 w=(0.286,0.286), b=0.714, SV={(0.5,0.5), (-3,-3)}와 정확히 일치")

w = [0.28571429 0.28571429]   b = 0.7142857142857144
서포트 벡터: [[-3.0, -3.0], [0.5, 0.5]]
본문 예상값 w=(0.286,0.286), b=0.714, SV={(0.5,0.5), (-3,-3)}와 정확히 일치


## 7. 서포트 벡터 아닌 점을 멀리 옮겨도 \(w,b\)는 변하지 않는다

반대 방향 실험: \((4,3)\)을 원점에서 **훨씬 먼** \((100,100)\)으로
밀어낸 뒤 재학습한다. \((4,3)\)은 서포트 벡터가 아니었으므로
\(w,b\)는 소수점까지 동일해야 한다 — "마진 최대화는 전체 분포가
아니라 경계에 가장 가까운 소수의 점만으로 정해진다"는 본문의 핵심.

마지막으로 같은 점 이동을 **로지스틱회귀**에도 해본다: 모든 점이
손실에 기여하므로 \(w\)가 달라진다 — SVM과 로지스틱회귀의 근본적
차이(이상치 견고성 트레이드오프)를 직접 보는 실험.

In [7]:
from sklearn.linear_model import LogisticRegression

# (4,3) -> (100,100): SVM
X_far = X_sym.copy()
X_far[1] = [100.0, 100.0]
clf_f = SVC(kernel="linear", C=1.0).fit(X_far, y)
print(f"SVM  이동 전: w={clf.coef_[0]}  b={clf.intercept_[0]}")
print(f"SVM  이동 후: w={clf_f.coef_[0]}  b={clf_f.intercept_[0]}")
assert np.allclose(clf.coef_[0], clf_f.coef_[0], atol=1e-12)
assert np.allclose(clf.intercept_[0], clf_f.intercept_[0], atol=1e-12)
print("-> SVM: (4,3)은 서포트 벡터가 아니므로 w,b는 소수점까지 동일\n")

# 같은 이동을 로지스틱회귀에
lr_before = LogisticRegression().fit(X_sym, y)
lr_after  = LogisticRegression().fit(X_far, y)
print(f"LR   이동 전: w={np.round(lr_before.coef_[0], 4)}  b={lr_before.intercept_[0]:.4f}")
print(f"LR   이동 후: w={np.round(lr_after.coef_[0], 4)}  b={lr_after.intercept_[0]:.4f}")
assert not np.allclose(lr_before.coef_[0], lr_after.coef_[0], atol=1e-6)
print("-> 로지스틱회귀: 멀리 있는 점도 손실에 기여하므로 w가 변한다")
print("  (이상치가 SV가 안 되는 한 SVM은 무시, LR은 무한정 멀리까지 영향 — 서로 다른 트레이드오프)")

SVM  이동 전: w=[0.16666667 0.16666667]  b=-0.0
SVM  이동 후: w=[0.16666667 0.16666667]  b=-0.0
-> SVM: (4,3)은 서포트 벡터가 아니므로 w,b는 소수점까지 동일

LR   이동 전: w=[0.5396 0.5396]  b=0.0000
LR   이동 후: w=[0.5035 0.5329]  b=-0.1691
-> 로지스틱회귀: 멀리 있는 점도 손실에 기여하므로 w가 변한다
  (이상치가 SV가 안 되는 한 SVM은 무시, LR은 무한정 멀리까지 영향 — 서로 다른 트레이드오프)


## 정리: 다음으로

- **5.2절 (소프트 마진)**: 완벽히 분리 안 되는 데이터에서는 제약을
  \(\ge1\)이 아니라 \(\ge1-\xi\)로 풀어놓고 \(C\sum\xi\)를
  더한다 — \(C\)가 "마진을 얼마나 지키려 하는가"를 결정한다.
- **5.3절 (커널 트릭)**: 선형분리 안 되는 데이터에서는
  \(w^Tx\)를 \(\kappa(x,x')\)로 바꾼 뒤 **같은 마진 최대화
  문제**를 해의 공간에서 푼다 — 이 노트북의 \(w,b\) 검증 코드가
  커널 SVM에서도(결정 경계는 더 이상 초평면이 아니지만) 그대로
  서포트 벡터 개념을 확인하는 데 쓰인다.

이 노트북에서 본문의 4개 수치(손 계산 \(w=1/6\), 병진 \(b=-1/3\),
확인 문제 \(w=2/7\), 거리 \(1/\|w\|\) 측정)가 전부 코드로 검증됐으므로,
5.1절의 "기하학적 마진" 논리가 대수적으로만 아니라 수치적으로도
성립한다는 뜻이다.